# 68 — Human Evaluation Protocol
**Goal:** Design annotation guidelines and measure inter-annotator agreement.

Automated metrics measure how well a system matches a reference — but the reference itself must come from somewhere. This chapter covers the **gold standard** of NLP evaluation: humans. It lays out an annotation protocol for resume data (schema, guidelines, training, quality control) and introduces **inter-annotator agreement** — the statistical tool that tells you whether your human labels are trustworthy enough to serve as a gold set at all.

**Why it matters for resumes / ATS:** every metric in Chapters 64–67 is only as good as the labels it is computed against. If two human annotators cannot agree on where a skill mention starts or whether a bullet is STAR-compliant, then no automated score built on those labels means anything. A measurable agreement target (typically Cohen's kappa ≥ 0.8 on the training batch) is the quality gate that makes the whole evaluation stack — golden sets, A/B tests, hallucination checks — trustworthy.

## 1. Annotation Guidelines

Human evaluation fails silently without a written protocol. The code prints the four-part structure: (1) an **annotation schema** defining exactly what gets labeled — skill entities, section boundaries, and STAR-compliant bullet quality on a 1–5 scale; (2) a **guidelines document** with examples, edge cases, and rules for ambiguous cases; (3) **annotator training**, including jointly labeling 10 samples and discussing disagreements until agreement exceeds 0.8; and (4) ongoing **quality control** with 10% overlap between annotators, weekly calibration, and per-annotator drift tracking.

**What the code does:** prints the protocol as reference material for the chapter. The key idea: schema and guidelines come *first*, because ambiguous labels ("is '5 years' part of the skill mention?") are the main source of disagreement — and disagreement is measurable, so guidelines can be iterated until it shrinks.

**Try it:** write your own edge-case rule for "Python (pandas)" — is the parenthetical a separate skill? Whatever you decide, the guidelines must say so, or your annotators will each decide differently.

In [ ]:
print('''Human evaluation protocol for resume NLP:

1. Define annotation schema (what to label)
   - Skill entities: exact skill mention
   - Section boundaries: where each section starts
   - Bullet quality: STAR compliance (1-5)

2. Create guidelines document
   - Include examples and edge cases
   - Define ambiguous case rules
   - Provide reference materials

3. Annotator training
   - Annotate 10 sample items together
   - Discuss disagreements
   - Achieve >0.8 agreement before starting

4. Quality control
   - 10% overlap between annotators
   - Weekly calibration sessions
   - Track per-annotator drift''')

## 2. Inter-Annotator Agreement

If two annotators label the same items, their agreement measures label quality. **Simple agreement** (fraction of identical labels) is intuitive but misleading: two annotators who both just guess the most common label will agree by chance. **Cohen's kappa** corrects for that, computing agreement *beyond chance*: `(observed - expected_chance) / (1 - expected_chance)`. The code applies the standard scale — ≥0.8 almost perfect, ≥0.6 substantial, ≥0.4 moderate (needs calibration), below that poor (revisit the guidelines).

**What the code does:** two annotators rate 10 bullets on the 0–5 quality scale. The run reports simple agreement 60% but kappa 0.444 — the gap is the chance correction at work: both annotators lean on the same common ratings, so about 28% agreement is expected by chance alone. Kappa lands in the "moderate — needs calibration" band: the labels are usable but not yet trustworthy enough to build a golden set on.

**Try it:** make the annotators disagree on exactly two more items and watch kappa collapse toward 0 — near-chance agreement is a red flag that the rubric is ambiguous.

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Two annotators rating bullet quality (0-5)
annotator_a = [4, 5, 3, 4, 2, 5, 3, 4, 5, 3]
annotator_b = [4, 4, 4, 3, 2, 5, 3, 4, 5, 4]

# Cohen's Kappa
kappa = cohen_kappa_score(annotator_a, annotator_b)
print(f"Cohen's Kappa: {kappa:.3f}")
if kappa >= 0.8:
    print("  Almost perfect agreement")
elif kappa >= 0.6:
    print("  Substantial agreement")
elif kappa >= 0.4:
    print("  Moderate agreement — needs calibration")
else:
    print("  Poor agreement — guidelines need revision")

# Percentage agreement
agreement = sum(1 for a, b in zip(annotator_a, annotator_b) if a == b) / len(annotator_a)
print(f"Simple agreement: {agreement:.0%}")

## 3. Annotation Workflow

Agreement measured on 10 items is a warm-up; production annotation needs an end-to-end workflow. The code prints a five-phase plan for a 200-resume annotation project: **PREP** (schema, guidelines, tooling such as Doccano, Label Studio, or Prodigy), **TRAIN** (20 shared samples annotated together, then 20 annotated separately and measured — iterate the guidelines until kappa > 0.8 *before* scaling up), **ANNOTATE** (split the remaining 160 samples, keep a 10% overlap for ongoing quality control, check agreement weekly), **ANALYZE** (compile the golden dataset, compute per-category metrics, identify systematic errors), and **ITERATE** (fix systematic issues, retrain annotators, expand the dataset).

**What the code does:** prints the plan, but the structure is the teaching: training and analysis bookend the annotation itself, and the 10% overlap is a continuous quality sensor, not an afterthought. The golden dataset this workflow produces is exactly what Chapter 67's A/B harness and Chapter 64's metrics consume.

**Try it:** note how the numbers scale — 20 + 20 training, 160 production, 10% overlap is the smallest project that still measures annotator drift.

In [ ]:
print('''Complete annotation workflow:
1. PREP
   - Define schema + guidelines
   - Select 200 representative resumes
   - Set up annotation tool (Doccano, Label Studio, Prodigy)

2. TRAIN
   - 20 samples: annotate together, discuss
   - 20 samples: annotate separately, measure agreement
   - Iterate guidelines until kappa > 0.8

3. ANNOTATE
   - Split remaining 160 samples
   - 10% overlap for ongoing QC
   - Weekly agreement check

4. ANALYZE
   - Compile golden dataset
   - Calculate per-category metrics
   - Identify systematic errors
   
5. ITERATE
   - Fix systematic issues
   - Retrain annotators
   - Expand dataset''')

## Summary: Human evaluation is the gold standard. Cohen's Kappa measures agreement beyond chance.

**Labels are only as trustworthy as the humans who made them — measure that trust before building on it.**

Simple agreement flatters; kappa, by correcting for chance, exposes whether annotators are genuinely following the same rubric. The 60%-vs-0.444 gap in this chapter's example is the entire lesson: raw agreement can look acceptable while the labels remain too noisy for a golden set. A protocol with schema, guidelines, training, and overlap QC turns individual judgments into a reusable evaluation resource — the gold sets that every earlier metric and the A/B harness depend on. The final chapter of this block zooms out from accuracy to production health: latency profiling, because a perfect evaluator is worthless if the pipeline it guards is too slow to serve.